<img src="images/nn.jpg" width="450" height="300">

## Forward Propagation

<img src="images/weights.jpg" width="600" height="500">

<img src="images/hidden.jpg" width="600" height="500">

 Verilen 1 input $[x_1,x_2,x_3]$ için;

<img src="images/nn-1sample.jpg" width="300" height="150">


m input verisi için düşünürsek; 

for i = 1 to m: 

$ \quad z^{1(i)} = W^1 a^{0(i)} + b^1 $

$ \quad a^{1(i)} = \sigma(z^{1(i)}) $

$ \quad z^{2(i)} = W^2 a^{1(i)} + b^2 $

$ \quad a^{2(i)} = \sigma(z^{2(i)}) $

<img src="images/nn-all.jpg" width="600" height="300">

## Backpropagation

$$ Z^1 = W^1 A^0 + b^1 \quad Z^2 = W^2 A^1 + b^2 $$
$$ A^1 = \sigma (Z^1)  \quad A^2 = \sigma (Z^2) = Y $$

Loss Function: $ L(\textbf y,a^{[2]}) = (-\textbf y \log a^{[2]})- (1-\textbf y) \log(1-a^{[2]})$

Cost Function $\frac{1}{m} \sum L(\textbf y,a^{[2]})$

Hidden layer ve output layer arasındaki weightlerin gradientini hesaplamak için;

$$ \frac{\partial E}{\partial w^{[2]}} = \frac{\partial E}{\partial a^{[2]}} \frac{\partial a^{[2]}}{\partial z^{[2]}} \frac{\partial z^{[2]}}{\partial w^{[2]}} $$

$$ \frac{\partial E}{\partial a^{[2]}} = \frac{-y}{a^{[2]}} + \frac{1-y}{1-a^{[2]}} $$
$$ \frac{\partial a^{[2]}}{\partial z^{[2]}} = a^{[2]} (1-a^{[2]}) $$
$$ \frac{\partial z^{[2]}}{\partial w^{[2]}} = a^{[1]}  $$

$$ \frac{\partial E}{\partial w^{[2]}} = \frac{1}{m} (a^{[2]}-y) a^{[1]}$$

Input layer ve hidden layer arasındaki weightlerin gradientini hesaplamak için;

$$ \frac{\partial E}{\partial w^{[1]}} = \frac{\partial E}{\partial a^{[2]}} \frac{\partial a^{[2]}}{\partial z^{[2]}} \frac{\partial z^{[2]}}{\partial a^{[1]}} \frac{\partial a^{[1]}}{\partial z^{[1]}} \frac{\partial z^{[1]}}{\partial w^{[1]}} $$

$$ \frac{\partial E}{\partial w^{[1]}} = w^{[2]} a^{[0]} * a^{[1]}(1-a^{[1]})$$

## ANN Code

In [1]:
import sklearn.datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from matplotlib import pyplot
import numpy as np

def loadDataset():
    # LOAD DATASET
    iris = sklearn.datasets.load_iris()
    X = iris.data  #shape 150,4
    y = iris.target #shape 150,1
    
    # Preprocess Data - labels to onehot encoding
    y = OneHotEncoding(y) # shape 150,3
    
    # Split data - X_train : 112,4  X_test : 38,4
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.50, random_state=42, shuffle =True)
    
    return X_train, X_test, y_train, y_test

def OneHotEncoding(y):
    classes = np.unique(y, return_counts=False)
    encodedY = np.zeros((y.shape[0],classes.size))
    for i in range(y.shape[0]):
        encodedY[i][y[i]] = 1
    return encodedY

def networkInitialize(in_neurons, h_neurons, y_neurons): # parameters: 5,4,3
    weights_in = np.random.rand(h_neurons, in_neurons)
    weights_h = np.random.rand(y_neurons,h_neurons)
    bias_in = np.random.rand(h_neurons, 1)
    bias_h = np.random.rand(y_neurons,1)
    params = {"W_in":weights_in, "W_h":weights_h, "b_in":bias_in, "b_h":bias_h}

    return params

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    return sigmoid(z)*(1 - sigmoid(z))

def forwardProp(X,params):
    z1 = np.dot(params["W_in"],X) + params["b_in"] # shape 3,112
    a1 = sigmoid(z1) # shape 3,112

    z2 = np.dot(params["W_h"],a1) + params["b_h"] # shape 3,112
    a2 = sigmoid(z2) # shape 3,112
    
    forwardParams = {"A_i":X, "Z_h":z1, "A_h":a1, "Z_o":z2, "predictions":a2}
    
    return forwardParams
    
def backProp(y_train, learning_rate, params, forwardParams):
    sampleSize = y_train.shape[1] # 112
    dz2 = forwardParams["predictions"] - y_train # 3,112 - 3,112 
    dw2 = np.dot(dz2,forwardParams["A_h"].T)/sampleSize # shape 3,3
    db2 = np.sum(dz2, axis=1,keepdims = True)/sampleSize # shape 3,1
    
    dz1 = np.dot(params["W_h"].T,dz2)*sigmoid_derivative(forwardParams["A_h"]) # shape 3,112
    dw1 = np.dot(dz1,forwardParams["A_i"].T)/sampleSize # shape 3,4
    db1 = np.sum(dz1, axis=1,keepdims = True)/sampleSize # shape 3,1 
    
    # UPDATE WEIGHTS
    params["W_in"] = params["W_in"] - (learning_rate * dw1)
    params["W_h"] = params["W_h"] - (learning_rate * dw2)
    params["b_in"] = params["b_in"] - (learning_rate * db1)
    params["b_h"] = params["b_h"] - (learning_rate * db2)
    
    return params
    
def evalualte(y_actual, y_predicted):
    tp =0
    tn =0
    
    for i in range(y_actual.size):
        if y_actual[i] == y_predicted[i]:
            tp = tp + 1

    acc = (tp)/(y_actual.size)
    return acc
    
    
if __name__ == "__main__":
    X_train, X_test, y_train, y_test = loadDataset()
    # Adjust data to apply matrix multiplications
    X_train = X_train.T # shape 4,112
    X_test = X_test.T # shape 4,38
    y_train = y_train.T # shape 3,112
    y_test = y_test.T # shape 3,38    
    
    # Initialize network - neuron numbers including bias neuron
    params = networkInitialize(X_train.shape[0], 4, y_train.shape[0])
    
    for i in range(30000):
        forwardParams = forwardProp(X_train, params)
        params = backProp(y_train, 0.01, params, forwardParams)
        
    forwardParams = forwardProp(X_train, params)
    y_actual = np.argmax(y_train, axis =0)
    y_predicted = np.argmax(forwardParams["predictions"], axis =0)
    acc = evalualte(y_actual, y_predicted)
    print("Train Accuracy:",acc)
    
    forwardParams = forwardProp(X_test, params)
    y_actual = np.argmax(y_test, axis =0)
    y_predicted = np.argmax(forwardParams["predictions"], axis =0)
    acc = evalualte(y_actual, y_predicted)
    print("Test Accuracy:",acc)
    

Train Accuracy: 0.6533333333333333
Test Accuracy: 0.6933333333333334
